# Analysis and Forecasting of Electricity Prices (Germany, 2018–2025)
## I. Data Preprocessing

**Importing the required libraries**

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import openmeteo_requests
import requests_cache
from retry_requests import retry
import holidays

**Setting the visualization style**

In [3]:
def set_style():
    mpl.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Georgia'],
        'font.size': 12,
        'axes.titlesize': 12,
        'axes.labelsize': 12,
        'axes.grid': True,
        'grid.linestyle': '--',
        'grid.color': 'lightgray',
        'grid.alpha': 0.4,
        'figure.figsize': (10, 6),
        'lines.linewidth': 1.5,
        'legend.frameon': False,
        'legend.fontsize': 12,
        'legend.labelcolor': '#555555',
        'axes.edgecolor': '#555555',
        'axes.labelcolor': '#555555',
        'xtick.color': '#555555',
        'ytick.color': '#555555',
        'axes.spines.top': True,
        'axes.spines.bottom': True,
        'axes.spines.left': True,
        'axes.spines.top': True,
        'axes.linewidth': 0.6,
        'axes.prop_cycle': mpl.cycler(color=[
            '#4C72B0', '#55A868', '#C44E52',
            '#8172B2', '#CCB974', '#64B5CD'
        ])
    })
set_style()

### 1. Electricity price data
- **Data source**: Bundesnetzagentur | SMARD.de Bundesnetzagentur|SMARD.de (https://www.smard.de/en/downloadcenter/download-market-data/)
- **Bidding zone**: BZN|DE-LU (Germany and Luxembourg)  
- **Period**: October 1, 2018 – October 19, 2025  
- **Frequency**: Weekly
- **Description**: The price series represents the arithmetic mean of all Day-Ahead market auction prices for each individual week

In [2]:
file1 = 'Day-ahead_prices_201810010000_202511100000_Week.csv'
cols1 = {
    'Start date': 'week_start',
    'Germany/Luxembourg [€/MWh] Calculated resolutions': 'elec_price'
}

df = pd.read_csv(
    file1,
    sep=';',
    usecols=['Start date', 'Germany/Luxembourg [€/MWh] Calculated resolutions']
).rename(columns=cols1)

print(df)

       week_start elec_price
0     Oct 1, 2018      51.45
1     Oct 8, 2018      54.53
2    Oct 15, 2018      64.00
3    Oct 22, 2018      47.71
4    Oct 29, 2018      46.43
..            ...        ...
366   Oct 6, 2025     103.27
367  Oct 13, 2025     116.63
368  Oct 20, 2025          -
369  Oct 27, 2025          -
370   Nov 3, 2025          -

[371 rows x 2 columns]


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371 entries, 0 to 370
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   week_start  371 non-null    object
 1   elec_price  371 non-null    object
dtypes: object(2)
memory usage: 5.9+ KB


In [5]:
df['week_start'] = pd.to_datetime(df['week_start'], errors='coerce')
df['elec_price'] = pd.to_numeric(df['elec_price'], errors='coerce')

In [6]:
df = df.dropna(subset=['week_start', 'elec_price']).reset_index(drop=True)

In [7]:
print(df)

    week_start  elec_price
0   2018-10-01       51.45
1   2018-10-08       54.53
2   2018-10-15       64.00
3   2018-10-22       47.71
4   2018-10-29       46.43
..         ...         ...
363 2025-09-15       50.97
364 2025-09-22       90.93
365 2025-09-29       79.17
366 2025-10-06      103.27
367 2025-10-13      116.63

[368 rows x 2 columns]


### 2. Natural Gas Future Price Data (TTF)
- **Data source**: Investing.com (https://www.investing.com/commodities/dutch-ttf-gas-c1-futures-historical-data)
- **Period**: October 1, 2018 – October 19, 2025
- **Frequency**: Weekly
- **Description**: This variable captures the price dynamics of the front-month futures contract for natural gas traded at the European Dutch TTF hub. Each observation corresponds to the weekly settlement price of the respective contract in the last trading session of the week.

In [8]:
file2 = 'Dutch TTF Natural Gas Futures Historical Data.csv'

cols2 = {
    'Date': 'week_start',
    'Price': 'nat_gas_price'
}

nat_gas_df = pd.read_csv(
    file2,
    sep=',',
    usecols=['Date', 'Price']
).rename(columns=cols2)

print(nat_gas_df)

     week_start  nat_gas_price
0    10/19/2025         32.016
1    10/12/2025         31.816
2    10/05/2025         32.170
3    09/28/2025         31.438
4    09/21/2025         32.697
..          ...            ...
363  11/04/2018         24.325
364  10/28/2018         24.285
365  10/21/2018         24.280
366  10/14/2018         26.860
367  10/07/2018         26.630

[368 rows x 2 columns]


In [9]:
nat_gas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   week_start     368 non-null    object 
 1   nat_gas_price  368 non-null    float64
dtypes: float64(1), object(1)
memory usage: 5.9+ KB


In [10]:
nat_gas_df['week_start'] = pd.to_datetime(nat_gas_df['week_start']) - pd.Timedelta(days=6)
nat_gas_df = nat_gas_df.sort_values('week_start').reset_index(drop=True)

In [11]:
print(nat_gas_df)

    week_start  nat_gas_price
0   2018-10-01         26.630
1   2018-10-08         26.860
2   2018-10-15         24.280
3   2018-10-22         24.285
4   2018-10-29         24.325
..         ...            ...
363 2025-09-15         32.697
364 2025-09-22         31.438
365 2025-09-29         32.170
366 2025-10-06         31.816
367 2025-10-13         32.016

[368 rows x 2 columns]


In [12]:
df = df.merge(nat_gas_df, on='week_start', how='left')

### 3. Meteorological Data
- **Data source**: Open-Meteo (https://open-meteo.com/en/docs/historical-weather-api)
- **Period**: October 1, 2018 – October 19, 2025
- **Frequency**: Daily
- **Description**: The dataset contains daily meteorological observations (wind speed and sunshine duration) for each of the 16 German federal states and two offshore areas in the North Sea and Baltic Sea. The data will be aggregated to weekly frequency and then averaged at the national level using weights proportional to the installed generation capacity of the respective power plant types (wind and solar) in each location.

In [13]:
file3 = 'open-meteo-48.75N9.29E251m.csv'

wind_df = pd.read_csv(
    file3,
    usecols=['location_id', 'time', 'wind_speed_10m_mean (m/s)']
).rename(columns={'wind_speed_10m_mean (m/s)': 'wind_speed'})

sun_df = pd.read_csv(
    file3,
    usecols=['location_id', 'time', 'sunshine_duration (s)']
).rename(columns={'sunshine_duration (s)': 'sunshine_duration'})

print(wind_df)
print(sun_df)

       location_id        time  wind_speed
0                0   10/1/2018        3.61
1                0   10/2/2018        3.74
2                0   10/3/2018        3.75
3                0   10/4/2018        1.66
4                0   10/5/2018        1.54
...            ...         ...         ...
46363           17  10/15/2025        6.16
46364           17  10/16/2025        8.15
46365           17  10/17/2025        7.58
46366           17  10/18/2025        7.45
46367           17  10/19/2025        4.97

[46368 rows x 3 columns]
       location_id        time  sunshine_duration
0                0   10/1/2018           10689.42
1                0   10/2/2018            8092.32
2                0   10/3/2018           12958.04
3                0   10/4/2018           36674.21
4                0   10/5/2018           37178.11
...            ...         ...                ...
46363           17  10/15/2025           31818.59
46364           17  10/16/2025           33060.91
46365   

In [14]:
sun_df = sun_df.drop(sun_df[sun_df['location_id'].isin([16, 17])].index)
print(sun_df['location_id'].unique())

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]


In [15]:
wind_df['time'] = pd.to_datetime(wind_df['time'], errors='coerce')
sun_df['time'] = pd.to_datetime(sun_df['time'], errors='coerce')

In [16]:
wind_df['week_start'] = wind_df['time'] - pd.to_timedelta(wind_df['time'].dt.weekday, unit='D')
sun_df['week_start'] = sun_df['time'] - pd.to_timedelta(sun_df['time'].dt.weekday, unit='D')

wind_df = (
    wind_df
    .groupby(['location_id', 'week_start'], as_index=False)
    .agg({'wind_speed': 'mean'})
)

sun_df = (
    sun_df
    .groupby(['location_id', 'week_start'], as_index=False)
    .agg({'sunshine_duration': 'sum'})
)

print(sun_df)

      location_id week_start  sunshine_duration
0               0 2018-10-01          176467.40
1               0 2018-10-08          233955.93
2               0 2018-10-15          239911.28
3               0 2018-10-22           96129.79
4               0 2018-10-29           93017.39
...           ...        ...                ...
5883           15 2025-09-15          220516.14
5884           15 2025-09-22          104562.57
5885           15 2025-09-29          184414.80
5886           15 2025-10-06           65980.19
5887           15 2025-10-13          159653.92

[5888 rows x 3 columns]


#### Installed generation capacity data
- **Data source**: Bundesnetzagentur (https://www.bundesnetzagentur.de/DE/Fachthemen/ElektrizitaetundGas/Versorgungssicherheit/Erzeugungskapazitaeten/Kraftwerksliste/start.html)
- **Note**: The values were taken from the file "Kraftwerksliste.xlsx".

In [17]:
wind_power_df = pd.DataFrame({
    'location_id': range(18),
    'wind_power': [1896, 2681.2, 16.6, 8979.7, 202.9, 125, 2640.6, 13193.7, 4113.2, 7803.9, 4156.6, 552, 1374.3, 5546.9, 8977.9, 1858, 7162.8, 1522.8]
})

sun_power_df = pd.DataFrame({
    'location_id': range(16),
    'solar_power': [11276.6, 24189.8, 334.3, 6980.4, 120.3, 165, 4186.9, 7983.7, 3720.5, 10856.6, 4548.7, 916.3, 4041.3, 4219.3, 3325.6, 2522.8]
})

In [18]:
wind_df = wind_df.merge(wind_power_df, on='location_id')
sun_df = sun_df.merge(sun_power_df, on='location_id')

In [19]:
wind_df = wind_df.assign(wind_w = wind_df.wind_speed * wind_df.wind_power)
sun_df = sun_df.assign(sun_w = sun_df.sunshine_duration * sun_df.solar_power)

In [20]:
wind = (
    wind_df
    .groupby('week_start', as_index=False)
    .agg(
        wind_speed = ('wind_w',
                      lambda x: x.sum() /
                      wind_df.loc[x.index, 'wind_power'].sum())
    )
)

In [21]:
sun = (
    sun_df
    .groupby('week_start', as_index=False)
    .agg(
        sunshine_duration = ('sun_w',
                             lambda x: x.sum() /
                             sun_df.loc[x.index, 'solar_power'].sum())
    )
)

In [22]:
weather_df = wind.merge(sun, on='week_start', how='left')

In [23]:
print(weather_df)

    week_start  wind_speed  sunshine_duration
0   2018-10-01    4.377025      184968.022149
1   2018-10-08    3.620766      231715.171238
2   2018-10-15    2.532683      210403.196832
3   2018-10-22    5.586504       79675.700568
4   2018-10-29    4.475013      105471.825791
..         ...         ...                ...
363 2025-09-15    5.059145      229848.664877
364 2025-09-22    3.899771      149812.382380
365 2025-09-29    4.131087      200897.884523
366 2025-10-06    3.838479       96410.688313
367 2025-10-13    3.061368      152772.209276

[368 rows x 3 columns]


In [24]:
weather_df['sunshine_duration'] = weather_df['sunshine_duration'] / 3600

In [25]:
df = df.merge(
    weather_df,
    on='week_start',
    how='left'
)

df.head()

,week_start,elec_price,nat_gas_price,wind_speed,sunshine_duration
0,2018-10-01,51.45,26.630,4.377025,51.380006
1,2018-10-08,54.53,26.860,3.620766,64.365325
2,2018-10-15,64.00,24.280,2.532683,58.445332
3,2018-10-22,47.71,24.285,5.586504,22.132139
4,2018-10-29,46.43,24.325,4.475013,29.297729


In [ ]:
df.to_csv('Day-ahead Prices_2015_2025_Week.csv', index=False, encoding='utf-8-sig') 